## 02 — Vector Search with SimlarEngine

In this notebook we index 500 Quora questions using locally-computed sentence embeddings and search by semantic similarity.

**What we cover**
- Encoding text with `sentence-transformers` and building a `SimlarEngine` index
- Incremental adds — growing the index after initial construction
- Single-query search via `search()` — returns ranked `SearchResult` objects
- Updating and deleting documents by ID
- Saving the index to disk and reloading it

In [ ]:
%pip install -q datasets sentence-transformers simlar

## Load and embed the dataset

We use [`BeIR/quora`](https://huggingface.co/datasets/BeIR/quora) — a corpus of 500,000 community questions.
We take the first 500 questions and encode them with `all-MiniLM-L6-v2` (384-dimensional embeddings).

The first 200 seed the initial index; the remaining 300 are added incrementally to show live growth.

In [ ]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from simlar import SimlarEngine

ds = load_dataset("BeIR/quora", "corpus", split="corpus[:500]")
all_texts = ds["text"]
all_ids   = ds["_id"]

model      = SentenceTransformer("all-MiniLM-L6-v2")
all_vectors = model.encode(all_texts, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

id_to_text = dict(zip(all_ids, all_texts))

print(f"Loaded {len(all_texts)} questions  |  embedding dim: {all_vectors.shape[1]}")
print(f"Sample: {all_texts[0][:100]}")

## Build the index

`SimlarEngine.add()` accepts a list of string IDs and a matching float32 matrix.

In [ ]:
idx = SimlarEngine()
idx.add(ids=list(all_ids[:200]), vectors=all_vectors[:200])

print(f"Index size : {idx.size}")
print(f"Is trained : {idx.is_trained}")

## Incremental add

Call `add()` again at any time to grow the index without rebuilding it.

In [ ]:
idx.add(ids=list(all_ids[200:]), vectors=all_vectors[200:])
print(f"After incremental add: {idx.size} documents")

## Single-query search and Multi-query search

`search(query, k)` accepts (n, dim) float32 array and returns a list of `SearchResult` objects, each with `.id`, `.rank`, and `.score`.

In [ ]:
QUERY_TEXT = "What is the best programming language to learn first?"
query_vec  = model.encode([QUERY_TEXT], normalize_embeddings=True).astype(np.float32)  # (1, 384)

print(f"Query: '{QUERY_TEXT}'\n")
for r in idx.search(query_vec, k=5):
    print(f"  rank={r.rank}  score={r.score:.4f}  →  {id_to_text[r.id]}")

In [ ]:
QUERIES = [
    "How do I invest in the stock market?",
    "What are the best ways to lose weight?",
    "How can I learn to speak Spanish quickly?",
]
query_matrix = model.encode(QUERIES, normalize_embeddings=True).astype(np.float32)  # (3, 384)

results = idx.search(query_matrix, k=3)
for i, query in enumerate(QUERIES):
    print(f"\nQuery: '{query}'")
    for r in results[i]:
        print(f"  rank={r.rank}  score={r.score:.4f}  →  {id_to_text[r.id]}")
  

## Update and delete

| Method | Effect |
|---|---|
| `update(ids, vectors)` | Replace the stored vector for existing IDs |
| `delete(ids)` | Remove documents from the index |

In [ ]:
print(f"Before: {idx.size} documents")

# Replace the embedding for the first document with a slightly perturbed version
rng       = np.random.default_rng(7)
perturbed = all_vectors[0:1] + rng.normal(0, 0.01, (1, all_vectors.shape[1])).astype(np.float32)
perturbed /= np.linalg.norm(perturbed)

idx.update(ids=[all_ids[0]], vectors=perturbed)
print(f"After update: {idx.size} documents  (size unchanged — update is in-place)")

# Delete the next two documents
idx.delete(ids=list(all_ids[1:3]))
print(f"After deleting 2 documents: {idx.size} documents")

## Save and reload

`save(directory)` writes the index to disk. `SimlarEngine.load(directory)` reconstructs it without re-encoding.
The reloaded index supports all the same operations — search, update, delete, and further adds.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    save_dir = str(Path(tmp) / "quora_index")

    idx.save(save_dir)
    print(f"Saved index to {save_dir}")
    print(f"Files: {[p.name for p in Path(save_dir).iterdir()]}")

    reloaded = SimlarEngine.load(save_dir)
    print(f"\nReloaded — size: {reloaded.size}  is_trained: {reloaded.is_trained}")

    # Verify search produces the same top result
    orig_top    = idx.search(query_vec, k=1)[0]
    reloaded_top = reloaded.search(query_vec, k=1)[0]
    match = orig_top.id == reloaded_top.id
    print(f"Top result matches original: {match}")
    print(f"  id={reloaded_top.id}  score={reloaded_top.score:.4f}  →  {id_to_text.get(reloaded_top.id)}")